In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import pandas as pd
import seaborn as sns

plt.rcParams.update({'font.size': 14})

In [ ]:
fig, axarr = plt.subplots(2, 3, figsize=(12, 9), gridspec_kw={'wspace': 0.3}, sharey=True)

model_names = { 
    'childs': 'Year + Month + P(Climate) (No regional terms)', 
    'lag1': 'Year + Month + Climate(lag 1)', 
    'lag_45': 'Year + Month + Climate(lag 1-5)\n(selected)', 
    'standard': 'Year + Month + Climate', 
    'with_immunity': 'Year + Month + Climate + Immunity', 
    'base': 'Year + Month',
    'socioeconomic': 'Year + Month + Climate + Socioeconomic',
    'lag1_cases': 'Year + Month + Climate + Cases(lag 1)'
}


##########
df = pd.concat([
    pd.read_csv(f'data/{model}_rsquared_aic_state_blockboot_presampled1000.csv') for model in model_names.keys()
])


df['model_name'] = df['model_name'].map(model_names)
df = df.reset_index().drop('index', axis=1)

df_rsq = df.copy()
df_aic = df.copy()
df_prsq = df.copy()

prsq_normed = df_prsq.groupby('bootstrap_iteration').apply(lambda x: 100 * (x['pseudo_r_squared'] / x.loc[x['model_name'] == 'Year + Month', 'pseudo_r_squared'].values[0] - 1)).reset_index().sort_values('level_1')
df_prsq = df_prsq.join(prsq_normed.set_index('level_1').rename(columns={'pseudo_r_squared': 'r_squared_normed'})[['r_squared_normed']])
df_prsq = df_prsq[df_prsq['model_name'] != 'Year + Month']

rsq_normed = df_rsq.groupby('bootstrap_iteration').apply(lambda x: 100 * (x['r_squared'] / x.loc[x['model_name'] == 'Year + Month', 'r_squared'].values[0] - 1)).reset_index().sort_values('level_1')
df_rsq = df_rsq.join(rsq_normed.set_index('level_1').rename(columns={'r_squared': 'r_squared_normed'})[['r_squared_normed']])
df_rsq = df_rsq[df_rsq['model_name'] != 'Year + Month']

aic_normed = df_aic.groupby('bootstrap_iteration').apply(lambda x: 100 * (x['aic'] / x.loc[x['model_name'] == 'Year + Month', 'aic'].values[0] - 1)).reset_index().sort_values('level_1')
df_aic = df_aic.join(aic_normed.set_index('level_1').rename(columns={'aic': 'aic_normed'})[['aic_normed']])
df_aic = df_aic[df_aic['model_name'] != 'Year + Month']

order = [
    'Year + Month + P(Climate) (No regional terms)', 
    'Year + Month + Climate(lag 1)', 'Year + Month + Climate', 
    'Year + Month + Climate + Socioeconomic', 
    'Year + Month + Climate + Immunity', 
    'Year + Month + Climate(lag 1-5)\n(selected)',
    'Year + Month + Climate + Cases(lag 1)',
]

sns.barplot(
    y='model_name', x='r_squared_normed', data=df_rsq, errorbar=('pi', 95), orient='h', ax=axarr[0, 0], color='C1', 
    order=order
)

sns.barplot(
    y='model_name', x='aic_normed', data=df_aic, errorbar=('pi', 95), orient='h', ax=axarr[0, 1], color='C1', 
    order=order
)

for ax in [axarr[0, 0], axarr[0, 1]]:
    ax.axvline(0, color='k', ls='--')
    ax.set_ylabel('')


axarr[0, 0].set_xlim([-60, 70])
axarr[0, 1].set_xlim([-30, 40])

axarr[0, 0].set_xlabel('R² Relative Change to Fixed Term\neffect only model (Year + Month) (%)')
axarr[0, 1].set_xlabel('AIC Relative Change to Fixed Term\neffect only model (Year + Month) (%)')


for i, model in enumerate(order):

    prsq = df_prsq[df_prsq['model_name'] == model]['pseudo_r_squared'].median()
    prsq_normed = df_prsq[df_prsq['model_name'] == model]['r_squared_normed'].median()

    rsq = df_rsq[df_rsq['model_name'] == model]['r_squared'].median()
    rsq_normed = df_rsq[df_rsq['model_name'] == model]['r_squared_normed'].median()

    aic = df_aic[df_aic['model_name'] == model]['aic'].median() / 1e7
    aic_normed = df_aic[df_aic['model_name'] == model]['aic_normed'].median()

    if rsq is not np.nan:        
        if rsq_normed > 0:
            axarr[0, 0].text(rsq_normed + 30, i, f'{rsq:.2f}', color='k', fontsize=12, va='center', ha='center')
        else:
            axarr[0, 0].text(rsq_normed - 40, i, f'{rsq:.2f}', color='k', fontsize=12, va='center', ha='center')

    if aic is not np.nan:
        if aic_normed > 0:
            axarr[0, 1].text(aic_normed + 20, i, f'{aic:.2f}', color='k', fontsize=12, va='center', ha='center')
        else:
            axarr[0, 1].text(aic_normed - 10, i, f'{aic:.2f}', color='k', fontsize=12, va='center', ha='center')


# Out of sample validation metrics
df = pd.read_csv(f'data/all_models_validation_metrics_5year_windows.csv')
df['r2'] = df['correlation'] ** 2

model_names = { 
    'childs': 'Year + Month + P(Climate) (No regional terms)', 
    'lag1': 'Year + Month + Climate(lag 1)', 
    'lag_45': 'Year + Month + Climate(lag 1-5)\n(selected)', 
    'no_month': 'Year + Climate', 
    'standard': 'Year + Month + Climate', 
    'with_immunity': 'Year + Month + Climate + Immunity', 
    'base': 'Year + Month',
    'socioeconomic': 'Year + Month + Climate + Socioeconomic',
    'lag1_cases': 'Year + Month + Climate + Cases(lag 1)',
}
df['model'] = df['model'].map(model_names)

order = [
    'Year + Month + P(Climate) (No regional terms)', 
    'Year + Month + Climate(lag 1)', 
    'Year + Month + Climate', 
    'Year + Month + Climate + Socioeconomic', 
    'Year + Month + Climate + Immunity', 
    'Year + Month + Climate(lag 1-5)\n(selected)',
    'Year + Month + Climate + Cases(lag 1)',
]

variables = ['pseudo_r2', 'r2', 'rmse']

for ax, column in zip(axarr[1], variables):
    df_copy = df.copy()

    rsq_normed = df.groupby('iteration').apply(lambda x: 100 * (x[column] / x.loc[x['model'] == 'Year + Month', column].values[0] - 1)).reset_index().sort_values('level_1')
    df_copy = df_copy.join(rsq_normed.set_index('level_1').rename(columns={column: 'r_squared_normed'})[['r_squared_normed']])

    df_copy = df_copy[df_copy['model'] != 'Year + Month']
    

    sns.boxplot(
        y='model', x='r_squared_normed', data=df_copy, ax=ax, color='C3',
        order=order, showfliers=False
    )

    ax.axvline(0, color='k', ls='--')
    ax.set_ylabel('')



for i, model in enumerate(order):

    for ax, column in zip(axarr[1], variables):

        df_copy = df.copy()
        value = df_copy[df_copy['model'] == model][column].median()

        rsq_normed = df.groupby('iteration').apply(lambda x: 100 * (x[column] / x.loc[x['model'] == 'Year + Month', column].values[0] - 1)).reset_index().sort_values('level_1')
        df_copy = df_copy.join(rsq_normed.set_index('level_1').rename(columns={column: 'r_squared_normed'})[['r_squared_normed']])

        df_copy = df_copy[df_copy['model'] != 'Year + Month']
        value_normed = df_copy[df_copy['model'] == model]['r_squared_normed'].median()

        if column == 'pseudo_r2':        
            if value_normed > 0:
                ax.text(value_normed + 10, i, f'{value:.2f}', color='k', fontsize=12, va='center', ha='center')
            else:
                ax.text(value_normed - 18, i, f'{value:.2f}', color='k', fontsize=12, va='center', ha='center')

        if column == 'r2':        
            ax.text(value_normed + 90, i, f'{value:.2f}', color='k', fontsize=12, va='center', ha='center')

        if column == 'rmse':        
            ax.text(value_normed - 35, i, f'{value:.1f}', color='k', fontsize=12, va='center', ha='center')


axarr[1, 0].set_xlim([-28, 18])
axarr[1, 1].set_xlim([-60, 130])
axarr[1, 2].set_xlim([-50, 30])

axarr[1, 0].set_xlabel('Pseudo-R² % Change Relative to Fixed Term\n Effect Only Model (Year + Month)')
axarr[1, 1].set_xlabel('R² % Change Relative to Fixed Term\n Effect Only Model (Year + Month)')
axarr[1, 2].set_xlabel('RMSE % Change Relative to Fixed Term\n Effect Only Model (Year + Month)')

fig.delaxes(axarr[0, 2])

plt.tight_layout()
plt.savefig('img/supplementary_metrics.pdf', dpi=300, bbox_inches='tight')